# Null and absence

## "No value" needs a type
A lot of the work in a decision engine is about absence. The vehicle
lookup returned nothing. The previous attempt has no transaction
key. The merchant name is missing. Each of these is a legitimate
domain state — not a programming error — and the type system either
lets you name it or it doesn't.

When the type system *doesn't*, you get `null`. And `null` is the
problem.

The concrete example we use throughout: **"vehicle not found."** It's
not an exception, it's not a bug, it's not a corrupt row. It's an
ordinary outcome of an ordinary classification. Five engines model
it five different ways.

## Normal C# — `null` everywhere, NRT off

In [ ]:
// normalcsharp-fuel-engine/src/FuelUploadEngine/Models/RowDecision.cs
public string Status;
public FuelTransaction Transaction;       // null when Status is Rejected / Fatal / Skipped
public Vehicle Vehicle;                   // null when vehicle lookup failed
public string SkipReason;
public string FatalMessage;

NRT is off in the `.csproj`. Every reference type is implicitly
nullable. "Vehicle not found" is signalled by:

1. `VehicleLookupStatus` string set to `"NotFound"`.
2. `Vehicle` property left `null`.
3. `Status` set to `"Rejected"`.
4. An error string `"VehicleNotFound:..."` pushed onto `Errors`.

Four implicit invariants encoded across four separately-typed fields.
The compiler enforces none of them. A debug `Console.WriteLine` in
the service reads `d.Vehicle.LicensePlate` unconditionally — for a
`Rejected` row that's `null.LicensePlate`, and the whole batch dies
with NullReferenceException. (See footgun #1 in [chapter
03](../part1/03-seven-footguns.ipynb).)

`null` is the universal absence story. It also stands in for "I
forgot to populate this", "the deserialiser left it default", and
"this row failed before we got to that field." The same value, three
meanings.

## Idiomatic C# — NRT on, typed absence via DUs

In [ ]:
// csharp-fuel-engine/src/FuelUploadEngine/Domain/VehicleLookupResult.cs
public abstract record VehicleLookupResult
{
    private VehicleLookupResult() { }

    public sealed record Found(Vehicle Vehicle) : VehicleLookupResult;
    public sealed record NotFound(VehicleIdentifier Identifier) : VehicleLookupResult;
    public sealed record Ambiguous(VehicleIdentifier Identifier, IReadOnlyList<Vehicle> Candidates) : VehicleLookupResult;
    public sealed record Unavailable(FatalError Error) : VehicleLookupResult;
}

`<Nullable>enable</Nullable>` is set in the `.csproj`. Every domain
property is non-nullable; the compiler enforces that you never assign
`null` to one without an explicit `?`. "Vehicle not found" is a
**named sum-type case** — `VehicleLookupResult.NotFound` carrying
the requested identifier so the caller knows *which* vehicle wasn't
found. It is not the absence of a `Vehicle` value; it's the
**presence** of a typed "no-vehicle" value.

Compare:

In [ ]:
// in the classifier:
return lookup switch
{
    VehicleLookupResult.Found f      => /* use f.Vehicle */,
    VehicleLookupResult.NotFound nf  => Reject(nf.Identifier),
    VehicleLookupResult.Ambiguous a  => Reject(a.Identifier, a.Candidates),
    VehicleLookupResult.Unavailable u => Fatal(u.Error),
    _ => throw new InvalidOperationException()
};

No null checks. The compiler tracks which arm gives access to which
fields. The only remaining null story is **at the boundary**, where
DTO records have nullable string fields because JSON might omit them
(`string? VehicleId`, `string? UploadMode`). The mapper's job is to
collapse those nulls into typed parse errors before the domain ever
sees them.

## F# — no null in F# types, `Option<'T>` for absence

In [ ]:
[<RequireQualifiedAccess>]
type VehicleLookupResult =
    | Matched of Vehicle
    | NotFound
    | Ambiguous of Vehicle list
    | Fatal of FatalProcessingError

F# records, unions, and tuples **cannot be null** as far as F# is
concerned — assigning `null` to a value of an F# type is a compile
error. Absence has a dedicated type: `Option<'T> = Some of 'T | None`.
The classifier never sees `null`; it pattern-matches on a union or
unwraps an `Option`.

The exception, as ever, is `[<CLIMutable>]`:

In [ ]:
[<CLIMutable>]
type FuelUploadRowDto =
    { VehicleKey: string
      MerchantName: string
      /* ... */ }

The CLR sees a parameterless constructor and writable string
properties. A JSON deserialiser can produce a `FuelUploadRowDto`
where `VehicleKey = null`. F#'s mapper has to *defend* the boundary
with `String.IsNullOrWhiteSpace` checks — the only null-handling code
in the engine. Once past `Interop.fs`, no F# value can be `null`
again.

This is also why `Option` is more visible at the boundary than
inside the domain — the inside is null-free *by language*, but the
boundary has to bridge between "the wire might be null" and "the
domain is total."

## Haskell — no null, `Maybe a` and `Either e a`

In [ ]:
-- haskell-fuel-engine/src/FuelUpload/Domain/Vehicle.hs
data VehicleLookupResult
  = VehicleFound Vehicle
  | VehicleMissing Registration
  | VehicleLookupFatal FatalError
  deriving stock (Eq, Show)

Haskell has no `null`. The closest concept is `Maybe a = Nothing |
Just a`, which is just an ADT like any other. There is no
`[<CLIMutable>]` equivalent because there is no CLR to interop with;
the boundary uses JSON libraries that parse straight into ADTs,
producing `Maybe String` for optional fields rather than nullable
strings.

For lookups, "not found" is `VehicleMissing Registration` — a named
constructor with the offending registration as payload. For
operations that can fail, `Either e a` carries the error or the
success. The repository boundary in V3 uses exactly this:

In [ ]:
newtype VehicleRepository = VehicleRepository
  { repositoryLookupVehicle :: Registration -> Either VehicleRepositoryError VehicleLookupResult
  }

`Either` is the same shape as `Result` in F#/Rust. Nothing here is
`null`-shaped.

## Rust — no null, `Option<T>` and `Result<T, E>`

In [ ]:
// rust-fuel-engine/src/domain/vehicle.rs
#[derive(Debug, Clone, PartialEq, Eq)]
pub enum VehicleLookupResult {
    Found(Vehicle),
    NotFound { requested: VehicleRef },
    Ambiguous { requested: VehicleRef, matches: Vec<VehicleId> },
    Fatal(FatalError),
}

Rust has no `null` and no implicit nullable references. Absence is
`Option<T> = Some(T) | None`. Fallibility is `Result<T, E> = Ok(T) |
Err(E)`. "Vehicle not found" is a named variant of
`VehicleLookupResult` carrying the requested `VehicleRef` — the same
shape as idiomatic C#.

At the boundary, deserialisers produce `Option<String>` for fields
that might be missing in the JSON. The mapper turns those into typed
parse errors or `None` for genuinely-optional fields. The classifier
never sees an `Option<Vehicle>` that means "vehicle missing"; if the
vehicle is missing, the classifier sees `VehicleLookupResult::NotFound
{ requested }`.

## "Vehicle not found", lined up

| Engine | How "vehicle not found" is modelled | What the consumer sees |
|---|---|---|
| normal C# | `Vehicle` property = `null`, `VehicleLookupStatus = "NotFound"`, `Status = "Rejected"`, error string in list | a mutable `RowDecision` with conventionally-null fields; compile-time null analysis off; debug-print bug strikes here |
| idiomatic C# | `VehicleLookupResult.NotFound(VehicleIdentifier)` — typed sum-type case | exhaustive switch arm; NRT enforces no spurious null |
| F# | `VehicleLookupResult.NotFound` — DU case with no payload | pattern match arm; `null` not assignable to the type |
| Haskell | `VehicleMissing Registration` — ADT constructor with payload | case expression arm; `null` not a concept |
| Rust | `VehicleLookupResult::NotFound { requested: VehicleRef }` — enum variant with named field | `match` arm; no null in the language |

## Where nulls actually live in each engine

Even in the engines that "have no null," there's a boundary story.

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 2417.046875 164" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381590151_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381590151_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381590151_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381590151_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381590151_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381590151_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"></g><g class="edgeLabels"></g><g class="nodes"><g class="root" transform="translate(0, 0)"><g class="clusters"><g class="cluster" id="rust" data-look="classic"><rect style="" x="8" y="8" width="335" height="148"></rect><g class="cluster-label" transform="translate(159.046875, 8)"><foreignobject width="32.90625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Rust
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"></g><g class="edgeLabels"></g><g class="nodes"><g class="node default" id="flowchart-r1-8" transform="translate(175.5, 82)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
no null anywhere; Option at boundary
</p>
</span>
</div>
</foreignobject></g></g></g></g><g class="root" transform="translate(385, 0)"><g class="clusters"><g class="cluster" id="haskell" data-look="classic"><rect style="" x="8" y="8" width="335" height="148"></rect><g class="cluster-label" transform="translate(149.265625, 8)"><foreignobject width="52.46875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Haskell
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"></g><g class="edgeLabels"></g><g class="nodes"><g class="node default" id="flowchart-h1-7" transform="translate(175.5, 82)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
no null anywhere; Maybe at boundary
</p>
</span>
</div>
</foreignobject></g></g></g></g><g class="root" transform="translate(770, 0)"><g class="clusters"><g class="cluster" id="fsharp" data-look="classic"><rect style="" x="8" y="8" width="708.375" height="148"></rect><g class="cluster-label" transform="translate(352.8515625, 8)"><foreignobject width="18.671875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
F#
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M305.5,82L317.309,82C329.117,82,352.734,82,375.685,82C398.635,82,420.919,82,432.061,82L443.203,82" id="L_f1_f2_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_f1_f2_0" data-points="W3sieCI6MzA1LjUsInkiOjgyfSx7IngiOjM3Ni4zNTE1NjI1LCJ5Ijo4Mn0seyJ4Ijo0NDcuMjAzMTI1LCJ5Ijo4Mn1d" marker-end="url(#mermaid-1779381590151_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(376.3515625, 82)"><g class="label" data-id="L_f1_f2_0" transform="translate(-33.3515625, -12)"><foreignobject width="66.703125" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
Interop.fs
</p>
</span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-f1-4" transform="translate(175.5, 82)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
CLIMutable DTO can be deserialised to null
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-f2-6" transform="translate(563.0390625, 82)"><rect class="basic label-container" style="" x="-115.8359375" y="-27" width="231.671875" height="54"></rect><g class="label" style="" transform="translate(-85.8359375, -12)"><rect></rect><foreignobject width="171.671875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
domain: no null possible
</p>
</span>
</div>
</foreignobject></g></g></g></g><g class="root" transform="translate(1528.375, 12)"><g class="clusters"><g class="cluster" id="idiomatic" data-look="classic"><rect style="" x="8" y="8" width="604.96875" height="124"></rect><g class="cluster-label" transform="translate(266.46875, 8)"><foreignobject width="88.03125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
idiomatic C#
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M277.703,70L288.474,70C299.245,70,320.786,70,341.661,70C362.536,70,382.745,70,392.849,70L402.953,70" id="L_i1_i2_0" class="edge-thickness-normal edge-pattern-dotted edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_i1_i2_0" data-points="W3sieCI6Mjc3LjcwMzEyNSwieSI6NzB9LHsieCI6MzQyLjMyODEyNSwieSI6NzB9LHsieCI6NDA2Ljk1MzEyNSwieSI6NzB9XQ==" marker-end="url(#mermaid-1779381590151_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel" transform="translate(342.328125, 70)"><g class="label" data-id="L_i1_i2_0" transform="translate(-27.125, -12)"><foreignobject width="54.25" height="24">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel">
<p>
mapper
</p>
</span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-i1-1" transform="translate(161.6015625, 70)"><rect class="basic label-container" style="" x="-116.1015625" y="-27" width="232.203125" height="54"></rect><g class="label" style="" transform="translate(-86.1015625, -12)"><rect></rect><foreignobject width="172.203125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
only DTO <code>string?</code> fields
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-i2-3" transform="translate(491.2109375, 70)"><rect class="basic label-container" style="" x="-84.2578125" y="-27" width="168.515625" height="54"></rect><g class="label" style="" transform="translate(-54.2578125, -12)"><rect></rect><foreignobject width="108.515625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
domain: no null
</p>
</span>
</div>
</foreignobject></g></g></g></g><g class="root" transform="translate(2183.34375, 12)"><g class="clusters"><g class="cluster" id="normal" data-look="classic"><rect style="" x="8" y="8" width="217.703125" height="124"></rect><g class="cluster-label" transform="translate(79.9453125, 8)"><foreignobject width="73.8125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
normal C#
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"></g><g class="edgeLabels"></g><g class="nodes"><g class="node default" id="flowchart-n1-0" transform="translate(116.8515625, 70)"><rect class="basic label-container" style="" x="-71.3515625" y="-27" width="142.703125" height="54"></rect><g class="label" style="" transform="translate(-41.3515625, -12)"><rect></rect><foreignobject width="82.703125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
everywhere
</p>
</span>
</div>
</foreignobject></g></g></g></g></g></g></g>
</svg>
</div>

The shorter that bottom row, the cheaper your null-handling code.

Normal C# pays everywhere. Idiomatic C# pays at the DTO. F# pays at
`[<CLIMutable>]`. Haskell and Rust don't pay — `Maybe`/`Option` is
the same mechanism throughout, with no special boundary syntax.

The thing the ladder is really trading off here is **whether
"missing" is a value or a hole**. In a hole-based system, every
function has to ask "is this thing actually there?" In a value-based
system, the type that *might* be missing names the missing case
explicitly, and the function asks "which case is this?"

The second question has an answer the compiler can check. The first
one doesn't.

For the journey version, see [chapter 4 — idiomatic
C#](../part1/04-idiomatic-csharp.ipynb) on NRT and the typed-absence
trick, and [chapter 5 — F# native](../part1/05-fsharp-native.ipynb)
for the first language on the ladder where null genuinely cannot
exist inside the domain.